In [ ]:
from __future__ import annotations

import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

from notebooks.utils.classification_experiment_utils import (
    DATASETS, SOURCE_SPLITS, TRAINING_REGIMES, availability_row,
    build_feature_parent_cohorts, export_result_frames, load_training_episode, release_frames,
)
from notebooks.utils.feature_aggregation_utils import (
    FeatureSet, ensure_prepared_partition, prepared_partition_path, load_prepared_partition,
)
from notebooks.utils.model_training_utils import (
    MODEL_FAMILIES as AVAILABLE_MODEL_FAMILIES, fit_feature_vector_model, save_model_artifact,
    score_feature_vector_model,
)
from notebooks.utils.classification_utils import evaluate_scores

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
RUN_ID = "feature-vector-classification-2026-09-11"
SCORER_LLMS = ("gemma_3_1b_it", "gemma_3_270m_it", "littlelamb", "smollm2_360m")
DATASETS = {"Kaggle": "kaggle_train", "MAGE": "mage_train", "Outfox": "outfox_chatgpt_train", "BBC": "bbc_news_content"}
SOURCE_SPLITS = {"Kaggle": ("HG", "MG"), "MAGE": ("HG", "MG"), "Outfox": ("HG", "MG"), "BBC": ("HG",)}
TRAINING_REGIME = "Mixed"
TRAINING_REGIMES = {"Kaggle": ("Kaggle",), "MAGE": ("MAGE",), "Outfox": ("Outfox",), "Mixed": ("Kaggle", "MAGE", "Outfox"), "Mixed+BBC": ("Kaggle", "MAGE", "Outfox", "BBC")}
WINDOW_CONFIGURATIONS = ("whole_text", 32, 64, 128)
FEATURE_SETS = {
    "scalar_all": FeatureSet("scalar_all", ("d_score_top20", "perplexity", "prompt_mu_score", "prompt_sigma_score", "d_score_top20_coverage")),
    "scalar_compact": FeatureSet("scalar_compact", ("d_score_top20", "perplexity", "prompt_mu_score")),
}
MODEL_FAMILIES = ("logistic_regression",)
TRAINING_PARTITIONS = ("train", "validation")
COHORT_SEED, MAX_PER_SPLIT, STRIDE = 42, 500, 16


In [ ]:
preparation_rows, available = [], {}
for scorer_llm in SCORER_LLMS:
    for source_name, dataset in DATASETS.items():
        for split in SOURCE_SPLITS[source_name]:
            for feature_set in FEATURE_SETS.values():
                for window_configuration in WINDOW_CONFIGURATIONS:
                    path = prepared_partition_path(DATA_ROOT, scorer_llm=scorer_llm, dataset=dataset, split=split, feature_set=feature_set, window_configuration=window_configuration)
                    try:
                        partition_exists = path.exists()
                        status = ensure_prepared_partition(DATA_ROOT, PROCESSED_ROOT, scorer_llm=scorer_llm, dataset=dataset, split=split, feature_set=feature_set, window_configuration=window_configuration, stride=STRIDE)
                        preparation_rows.append({**availability_row(stage="preparation", status=status["status"], scorer_llm=scorer_llm, dataset=source_name, split=split, feature_set=feature_set.name, window_configuration=window_configuration, error=status.get("error", "")), "path": str(status["path"])})
                        if status["status"] == "available":
                            available[(scorer_llm, source_name, split, feature_set.name, str(window_configuration))] = Path(status["path"])
                    except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                        preparation_rows.append({**availability_row(stage="preparation", status="unavailable", scorer_llm=scorer_llm, dataset=source_name, split=split, feature_set=feature_set.name, window_configuration=window_configuration, error=error), "path": str(path)})
                        continue
preparation_status = pd.DataFrame(preparation_rows)


In [ ]:
training_metrics, training_coverage, run_summary, artifacts = [], [], [], []
training_sources = TRAINING_REGIMES[TRAINING_REGIME]
for scorer_llm in SCORER_LLMS:
    for feature_set_name, feature_set in FEATURE_SETS.items():
        for window_configuration in WINDOW_CONFIGURATIONS:
            source_frames, missing_sources = [], []
            for source_name in training_sources:
                dataset = DATASETS[source_name]
                paths = [available.get((scorer_llm, source_name, split, feature_set_name, str(window_configuration))) for split in SOURCE_SPLITS[source_name]]
                if any(path is None for path in paths):
                    missing_sources.append(source_name)
                    continue
                for split in SOURCE_SPLITS[source_name]:
                    path = available.get((scorer_llm, source_name, split, feature_set_name, str(window_configuration)))
                    try:
                        frame = load_prepared_partition(path, expected={"scorer_llm": scorer_llm, "dataset": dataset, "split": split, "feature_set": feature_set_name, "feature_columns": list(feature_set.source_columns), "window_configuration": "whole_text" if window_configuration == "whole_text" else str(window_configuration), "schema_version": 1, "stride": None if window_configuration == "whole_text" else STRIDE})
                        source_frames.append(frame.assign(dataset=dataset))
                    except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                        missing_sources.append(f"{source_name}: {error}")
            if missing_sources:
                release_frames(*source_frames)
                source_frames.clear()
                run_summary.append({"run_id": RUN_ID, "stage": "training", "status": "unavailable", "scorer_llm": scorer_llm, "dataset": "", "split": "", "feature_set": feature_set_name, "training_window": str(window_configuration), "evaluation_window": "", "model_family": "", "artifact_path": "", "error": f"missing training sources: {missing_sources}"})
                continue
            parent_frame = pd.concat(source_frames, ignore_index=True)[["dataset", "split", "text_id", "label"]]
            release_frames(*source_frames)
            source_frames.clear()
            try:
                cohorts = build_feature_parent_cohorts(parent_frame, seed=COHORT_SEED, max_per_split=MAX_PER_SPLIT)
                train, validation = load_training_episode(DATA_ROOT, scorer_llm=scorer_llm, datasets=tuple(DATASETS[name] for name in training_sources), feature_set=feature_set_name, window_configuration=window_configuration, feature_columns=feature_set.source_columns, train_cohort=cohorts["train"], validation_cohort=cohorts["validation"])
            except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                run_summary.append({"run_id": RUN_ID, "stage": "training", "status": "unavailable", "scorer_llm": scorer_llm, "dataset": ", ".join(DATASETS[name] for name in training_sources), "split": "", "feature_set": feature_set_name, "training_window": str(window_configuration), "evaluation_window": "", "model_family": "", "artifact_path": "", "error": str(error)})
                release_frames(parent_frame)
                continue
            for model_family in MODEL_FAMILIES:
                try:
                    fitted = fit_feature_vector_model(model_family, train, feature_columns=feature_set.source_columns, seed=COHORT_SEED)
                    scores = score_feature_vector_model(fitted, validation)
                    metrics = evaluate_scores(scores["label"], scores["mg_score"], 0.5)
                except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                    run_summary.append({"run_id": RUN_ID, "stage": "training", "status": "unavailable", "scorer_llm": scorer_llm, "dataset": ", ".join(DATASETS[name] for name in training_sources), "split": "", "feature_set": feature_set_name, "training_window": str(window_configuration), "evaluation_window": "", "model_family": model_family, "artifact_path": "", "error": str(error)})
                    continue
                training_metrics.append({"model_family": model_family, "scorer_llm": scorer_llm, "feature_set": feature_set_name, "training_regime": TRAINING_REGIME, "training_window": str(window_configuration), "partition": "validation", "sample_count": len(validation), **{key: metrics[key] for key in ("accuracy", "balanced_accuracy", "macro_f1", "roc_auc")}})
                training_coverage.append({"dataset": ", ".join(DATASETS[name] for name in training_sources), "split": "HG+MG", "class": "binary", "parent_count": len(cohorts["train"]) + len(cohorts["validation"]), "row_count": len(train) + len(validation), "availability_status": "available"})
                metadata = {"artifact_version": 1, "run_id": RUN_ID, "created_at": datetime.now(timezone.utc).isoformat(), "model_family": model_family, "scorer_llm": scorer_llm, "training_datasets": list(DATASETS[name] for name in training_sources), "training_regime": TRAINING_REGIME, "training_window": str(window_configuration), "feature_set": feature_set_name, "feature_columns": list(feature_set.source_columns), "cohort_seed": COHORT_SEED, "stride": None if window_configuration == "whole_text" else STRIDE, "train_row_count": len(train), "validation_row_count": len(validation), "train_parent_count": len(cohorts["train"]), "validation_parent_count": len(cohorts["validation"]), "train_hg_row_count": int(train["label"].eq(0).sum()), "train_mg_row_count": int(train["label"].eq(1).sum()), "validation_hg_row_count": int(validation["label"].eq(0).sum()), "validation_mg_row_count": int(validation["label"].eq(1).sum())}
                try:
                    artifact_path = save_model_artifact(DATA_ROOT, fitted, metadata=metadata, run_id=RUN_ID, training_metrics=pd.DataFrame(training_metrics[-1:]), training_coverage=pd.DataFrame(training_coverage[-1:]))
                except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                    run_summary.append({"run_id": RUN_ID, "stage": "training", "status": "unavailable", "scorer_llm": scorer_llm, "dataset": ", ".join(DATASETS[name] for name in training_sources), "split": "", "feature_set": feature_set_name, "training_window": str(window_configuration), "evaluation_window": "", "model_family": model_family, "artifact_path": "", "error": str(error)})
                    continue
                artifacts.append(artifact_path)
                run_summary.append({"run_id": RUN_ID, "stage": "training", "status": "available", "scorer_llm": scorer_llm, "dataset": ", ".join(DATASETS[name] for name in training_sources), "split": "HG+MG", "feature_set": feature_set_name, "training_window": str(window_configuration), "evaluation_window": "", "model_family": model_family, "artifact_path": str(artifact_path), "error": ""})
            release_frames(parent_frame, train, validation, scores)


In [ ]:
result_root = DATA_ROOT / "experiment-results" / "classification" / "runs"
result_root.mkdir(parents=True, exist_ok=True)
run_summary = pd.DataFrame(run_summary)
exports = export_result_frames(DATA_ROOT, run_id=RUN_ID, frames={
    "training_metrics": pd.DataFrame(training_metrics),
    "training_coverage": pd.DataFrame(training_coverage),
    "evaluation_metrics": pd.DataFrame(),
    "evaluation_predictions": pd.DataFrame(),
    "evaluation_coverage": pd.DataFrame(),
    "run_summary": run_summary,
})
preparation_status.to_csv(result_root / f"{RUN_ID}-preparation.csv", index=False)
run_summary.to_csv(result_root / f"{RUN_ID}-summary.csv", index=False)
exports
